# Step 2: Autonomous Runway FOD Detector Training
## Model Architecture: YOLO11 Nano (`yolo11n`) with Spatial Attention (C2PSA)

---
### Mission Objective
Train a state-of-the-art, real-time object detector optimized for runway Foreign Object Debris (FOD) inspection on an unmanned aerial vehicle (UAV).

### Assessment Task 2 Requirements:
- Train a justified detector (**YOLO11 Nano** chosen for its **C2PSA Spatial Attention** and ultra-low edge latency).
- Report & justify configuration: **image size, epochs, batch size, and augmentations**.
- Output: Trained weights (`best.pt`) with full training curves and configuration documented.


In [ ]:
import os
import sys
from pathlib import Path
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

from ultralytics import YOLO
import ultralytics
print(f"Ultralytics Version: {ultralytics.__version__}")


## 1. Training Configuration & Engineering Justification

| Hyperparameter | Value | Runway Engineering & Hardware Justification |
| :--- | :---: | :--- |
| **Model** | `yolo11n.pt` | ~2.6M params. Incorporates **C2PSA (Cross-Stage Partial with Spatial Attention)** to isolate tiny anomalous debris against vast runway tarmac textures. |
| **Input Resolution (`imgsz`)** | `640` | **Small-object preservation:** 54.2% of our dataset objects are $<32\times32$ px. A 640px input ensures fine details (screws, wires) survive downsampling in the P3 feature stride. |
| **Batch Size (`batch`)** | `16` | Optimized for 4GB RTX 2050 VRAM with Automatic Mixed Precision (`fp16`). Maintains stable Batch Normalization statistics without VRAM overflow. |
| **Epochs (`epochs`)** | `35` | Transfer learning from COCO pre-trained weights requires moderate epochs to adapt intermediate features to asphalt surfaces. |
| **Early Stopping (`patience`)**| `10` | Halts training if validation mAP fails to improve for 10 consecutive epochs to prevent overfitting on specific tarmac textures. |
| **Optimizer** | `AdamW` | Decoupled weight decay ($0.0005$) provides stable convergence on high-frequency tarmac textures (cracks, gravel). |
| **Learning Rate (`lr0`, `lrf`)**| `0.002` $\to$ `0.00002` | **Cosine Annealing (`cos_lr=True`)** enables smooth annealing into broad, generalizable loss minima. |
| **Mosaic Augmentation** | `1.0` | Combines 4 images into one, simulating **varying drone altitudes and camera zoom levels** (satisfying the extra points criteria). |
| **HSV Jitter** | $H=0.015, S=0.7, V=0.4$ | Simulates variable lighting—from harsh noon sun glare to dim overcast and dusk conditions. |
| **Rotational Invariance** | `degrees=15.0, fliplr=0.5, flipud=0.5` | Runway debris falls in arbitrary 360-degree orientations relative to the aircraft centerline or drone flight vector. |


In [ ]:
DATASET_YAML = Path('data/dataset.yaml').resolve()
assert DATASET_YAML.exists(), f"dataset.yaml not found at {DATASET_YAML}"

with open(DATASET_YAML, 'r') as f:
    config = yaml.safe_load(f)

print("Loaded Dataset Configuration:")
print(f"  Path: {config['path']}")
print(f"  Train: {config['train']}")
print(f"  Val:   {config['val']}")
print(f"  Test:  {config['test']}")
print(f"  Number of classes: {config['nc']}")
print(f"  Classes: {config['names']}")


In [ ]:
# Initialize YOLO11 Nano with pre-trained COCO weights
model = YOLO('yolo11n.pt')

# Training Hyperparameters dictionary
train_args = {
    'data': str(DATASET_YAML),
    'epochs': 35,
    'imgsz': 640,
    'batch': 16,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'optimizer': 'AdamW',
    'lr0': 0.002,
    'lrf': 0.01,
    'cos_lr': True,
    'weight_decay': 0.0005,
    'patience': 10,
    'save': True,
    'project': 'runs/detect',
    'name': 'fod_yolo11n_runway',
    'exist_ok': True,
    'pretrained': True,
    'amp': True,
    
    # Runway-specific augmentations
    'mosaic': 1.0,           # Simulates multiple drone altitudes / scales
    'mixup': 0.1,            # Handles semi-transparent/occluded debris
    'hsv_h': 0.015,          # Hue jitter
    'hsv_s': 0.7,            # Saturation jitter for wet tarmac reflections
    'hsv_v': 0.4,            # Brightness jitter for bright sun vs dim dusk
    'degrees': 15.0,         # Slight camera tilt / debris orientation
    'fliplr': 0.5,           # Horizontal flip
    'flipud': 0.5,           # Vertical flip
    'close_mosaic': 10,      # Disable mosaic for last 10 epochs for fine localization
    'plots': True,
    'verbose': True
}

print("Configured training parameters:")
for k, v in train_args.items():
    print(f"  {k}: {v}")


## 2. Launch Training Pipeline
Training executes with Automatic Mixed Precision (`amp=True`) on the NVIDIA RTX 2050 GPU. Checkpoints, validation confusion matrices, and PR curves are automatically logged under `runs/detect/fod_yolo11n_runway/`.


In [ ]:
# Execute Model Training
print("Starting YOLO11n Runway FOD Training...")
results = model.train(**train_args)
print("Training successfully completed!")


## 3. Training & Validation Performance Curves
Ultralytics automatically generates loss curves, mAP progression, and confusion matrices in the experiment directory. We inspect them below.


In [ ]:
RUN_DIR = Path('runs/detect/fod_yolo11n_runway')

results_png = RUN_DIR / 'results.png'
if results_png.exists():
    img = Image.open(results_png)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.title('YOLO11n Training & Validation Metrics (Loss, Precision, Recall, mAP50, mAP50-95)', fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print(f"Results image not found at {results_png}")

# Best model weights path
best_weights = RUN_DIR / 'weights' / 'best.pt'
print(f"Best Model Checkpoint: {best_weights} (Exists: {best_weights.exists()})")
